# 🧠 Redes Neurais MLP (Multi-Layer Perceptron) na Prática

**Aula prática — Deep Learning com PyTorch**

---

### 📚 Roteiro da aula

1. **O neurônio e o Perceptron** — de onde tudo começou
2. **MLP: arquitetura e funções de ativação** — empilhando neurônios
3. **Backpropagation** — como a rede aprende (intuição matemática)
4. **Laboratório PyTorch** — treinando no MNIST do zero
5. **Regularização e otimização** — combatendo overfitting
6. **🎯 Exercício final** — para fixar o conteúdo

---

> 💡 **Dica:** Este notebook é o pré-requisito conceitual para as CNNs. Se você entender bem o fluxo forward → loss → backward → step de um MLP, vai entender qualquer arquitetura de deep learning!

## 🔧 Setup inicial

Vamos importar as mesmas bibliotecas que usamos no notebook de CNNs. A única novidade é o `sklearn`, que vai nos ajudar a criar alguns datasets sintéticos para visualizar conceitos.

In [1]:
import numpy as np                    # Operações numéricas com arrays
import matplotlib.pyplot as plt       # Visualização de gráficos
import matplotlib.colors as mcolors   # Paletas de cores para os gráficos

# PyTorch
import torch                          # Núcleo do PyTorch (tensores e autograd)
import torch.nn as nn                 # Módulos para construir redes neurais
import torch.nn.functional as F       # Funções de ativação, loss, etc.
import torch.optim as optim           # Otimizadores (SGD, Adam, etc.)

# Torchvision para carregar o MNIST
import torchvision
import torchvision.transforms as transforms
from torchvision import datasets
from torch.utils.data import DataLoader

# sklearn — só para gerar datasets sintéticos de demonstração
from sklearn.datasets import make_moons, make_circles, make_classification
from sklearn.preprocessing import StandardScaler

# Dispositivo: GPU se disponível, senão CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo em uso: {device}')
print(f'Versão do PyTorch: {torch.__version__}')

# Seed para reprodutibilidade
torch.manual_seed(42)
np.random.seed(42)

ModuleNotFoundError: No module named 'torchvision'

---
# 🔬 Parte 1 — O Neurônio e o Perceptron

## Do neurônio biológico ao artificial

Um neurônio biológico recebe sinais pelos **dendritos**, processa no corpo celular e dispara um sinal pelo **axônio** — mas só se a estimulação for forte o suficiente (há um **limiar de ativação**).

O neurônio artificial imita isso com matemática simples:

$$z = w_1 x_1 + w_2 x_2 + \dots + w_n x_n + b = \mathbf{w}^T \mathbf{x} + b$$

$$\hat{y} = f(z)$$

Onde:
- $x_i$ são as **entradas** (features)
- $w_i$ são os **pesos** (o que o modelo aprende)
- $b$ é o **bias** (deslocamento — permite que o neurônio ative mesmo com entradas zero)
- $f$ é a **função de ativação** (introduz não-linearidade)
- $z$ é a **combinação linear** (também chamada de *pré-ativação*)

### O Perceptron

O **Perceptron** (Rosenblatt, 1958) é o neurônio artificial mais simples: usa uma função de ativação degrau.
- Se $z \geq 0$: saída = 1
- Se $z < 0$: saída = 0

Geometricamente, um perceptron define um **hiperplano** que separa duas classes no espaço de features.

In [ ]:
# ===== Visualizando o que um único neurônio faz =====

# Cria um dataset 2D linearmente separável (2 features → fácil de plotar)
# make_classification gera pontos de 2 classes que dá pra separar com uma reta
X, y = make_classification(
    n_samples=200,     # 200 pontos
    n_features=2,      # 2 features (para plotar no plano 2D)
    n_redundant=0,     # sem features redundantes
    n_informative=2,   # ambas as features são informativas
    random_state=1,
    n_clusters_per_class=1
)

# Normaliza os dados: média 0, desvio padrão 1
# Sempre faça isso! Facilita MUITO o treinamento.
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Converte para tensores PyTorch
X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32).unsqueeze(1)  # [200] → [200, 1]

print(f'Shape de X: {X_tensor.shape}')  # [200, 2] — 200 amostras, 2 features
print(f'Shape de y: {y_tensor.shape}')  # [200, 1] — rótulos 0 ou 1
print(f'Classes: {np.unique(y)}')       # [0, 1]

# --- Treina um único neurônio (perceptron) com sigmoide ---
# nn.Linear(2, 1) → pesos: w1, w2 e bias b
perceptron = nn.Sequential(
    nn.Linear(2, 1),   # z = w1*x1 + w2*x2 + b
    nn.Sigmoid()       # f(z) = 1 / (1 + e^-z)  → saída entre 0 e 1
)

# Otimizador SGD — atualiza pesos na direção do gradiente
otimizador = optim.SGD(perceptron.parameters(), lr=0.1)

# Loss: Binary Cross-Entropy — padrão para classificação binária
# BCE = -[y * log(ŷ) + (1-y) * log(1-ŷ)]
criterio = nn.BCELoss()

# Loop de treino simples — 200 épocas
losses = []
for epoch in range(200):
    otimizador.zero_grad()          # zera os gradientes
    pred = perceptron(X_tensor)     # forward pass
    loss = criterio(pred, y_tensor) # calcula o erro
    loss.backward()                 # backpropagation
    otimizador.step()               # atualiza os pesos
    losses.append(loss.item())

print(f'\nLoss final: {losses[-1]:.4f}')

# --- Visualização da fronteira de decisão ---
# A fronteira de decisão é onde o neurônio prevê 50% para cada classe
# (o hiperplano z = 0, ou seja, a reta w1*x1 + w2*x2 + b = 0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: curva de loss durante o treino
axes[0].plot(losses)
axes[0].set_xlabel('Época'); axes[0].set_ylabel('BCE Loss')
axes[0].set_title('Curva de aprendizado do Perceptron'); axes[0].grid(True)

# Plot 2: fronteira de decisão
# Cria uma grade de pontos para mapear a saída do neurônio em todo o espaço
xx, yy = np.meshgrid(np.linspace(-3, 3, 300), np.linspace(-3, 3, 300))
grade = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)

with torch.no_grad():
    Z = perceptron(grade).reshape(xx.shape).numpy()

axes[1].contourf(xx, yy, Z, levels=50, cmap='RdBu', alpha=0.7)  # fundo colorido
axes[1].contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2)  # fronteira
scatter = axes[1].scatter(X[:, 0], X[:, 1], c=y, cmap='RdBu', edgecolors='k', s=30)
axes[1].set_title('Fronteira de decisão (única reta)')
axes[1].set_xlabel('Feature 1'); axes[1].set_ylabel('Feature 2')
plt.colorbar(scatter, ax=axes[1], label='Classe')

plt.tight_layout()
plt.show()

## 🚫 O problema do XOR — por que um neurônio não é suficiente

O Perceptron funcionou bem com dados linearmente separáveis. Mas e quando a fronteira é **não-linear**?

O exemplo clássico é o **XOR** (OU exclusivo):

| $x_1$ | $x_2$ | $x_1$ XOR $x_2$ |
|--------|--------|------------------|
| 0 | 0 | **0** |
| 0 | 1 | **1** |
| 1 | 0 | **1** |
| 1 | 1 | **0** |

Nenhuma reta consegue separar os 1s dos 0s nesse padrão. Isso foi provado por Minsky e Papert em 1969 e quase matou o campo das redes neurais por mais de uma década! A solução veio com as **redes multicamadas**.

In [ ]:
# ===== Demonstrando o problema do XOR =====

# Os 4 pontos do XOR
X_xor = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
y_xor = torch.tensor([[0.], [1.], [1.], [0.]])  # saídas do XOR

# ---- Perceptron simples (1 neurônio) ----
perceptron_xor = nn.Sequential(nn.Linear(2, 1), nn.Sigmoid())
opt_simples = optim.Adam(perceptron_xor.parameters(), lr=0.1)

for _ in range(2000):
    opt_simples.zero_grad()
    loss = nn.BCELoss()(perceptron_xor(X_xor), y_xor)
    loss.backward()
    opt_simples.step()

print('=== Perceptron simples no XOR ===')
print(f'Loss final: {loss.item():.4f}  (nunca converge!)')
with torch.no_grad():
    print('Previsões:', perceptron_xor(X_xor).round().T)
    print('Esperado: ', y_xor.T)

# ---- MLP com 1 camada oculta ----
# Com 2 neurônios na camada oculta, a rede aprende a COMBINAR fronteiras lineares
# produzindo uma fronteira não-linear — e resolve o XOR!
mlp_xor = nn.Sequential(
    nn.Linear(2, 4),    # camada oculta: 4 neurônios
    nn.ReLU(),          # ativação não-linear — ESSENCIAL para quebrar a linearidade
    nn.Linear(4, 1),    # camada de saída
    nn.Sigmoid()
)
opt_mlp = optim.Adam(mlp_xor.parameters(), lr=0.1)

for _ in range(2000):
    opt_mlp.zero_grad()
    loss = nn.BCELoss()(mlp_xor(X_xor), y_xor)
    loss.backward()
    opt_mlp.step()

print('\n=== MLP (com camada oculta) no XOR ===')
print(f'Loss final: {loss.item():.6f}')
with torch.no_grad():
    print('Previsões:', mlp_xor(X_xor).round().T)
    print('Esperado: ', y_xor.T)

# ---- Visualiza as fronteiras de decisão lado a lado ----
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
xx, yy = np.meshgrid(np.linspace(-0.5, 1.5, 300), np.linspace(-0.5, 1.5, 300))
grade = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)

for ax, modelo, titulo in zip(axes,
                               [perceptron_xor, mlp_xor],
                               ['Perceptron simples (FALHA)', 'MLP com camada oculta (RESOLVE)']):
    with torch.no_grad():
        Z = modelo(grade).reshape(xx.shape).numpy()
    ax.contourf(xx, yy, Z, levels=50, cmap='RdBu', alpha=0.7)
    ax.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2)
    cores = ['blue', 'red', 'red', 'blue']  # 0 = azul, 1 = vermelho
    for i, (ponto, classe) in enumerate(zip(X_xor.numpy(), y_xor.numpy())):
        ax.scatter(ponto[0], ponto[1], c=cores[i], s=200, zorder=5, edgecolors='black')
        ax.annotate(f'({int(ponto[0])},{int(ponto[1])})→{int(classe[0])}',
                    (ponto[0]+0.05, ponto[1]+0.05), fontsize=9)
    ax.set_title(titulo); ax.set_xlabel('x1'); ax.set_ylabel('x2')

plt.tight_layout()
plt.show()

---
# 🏗️ Parte 2 — Arquitetura MLP e Funções de Ativação

## A arquitetura do MLP

O **Multi-Layer Perceptron** é uma rede com:

```
Entrada (n features)
    ↓
[Camada Oculta 1: h₁ neurônios] ← pesos W¹, bias b¹
    ↓  (ativação f)
[Camada Oculta 2: h₂ neurônios] ← pesos W², bias b²
    ↓  (ativação f)
    ...
[Camada de Saída: k neurônios]   ← pesos Wᴸ, bias bᴸ
    ↓  (ativação de saída: softmax, sigmoide, ou nenhuma)
Saída (predição)
```

**Pontos-chave:**
- ✅ Cada camada é **totalmente conectada** (*fully connected* ou *dense*): todo neurônio de uma camada conecta a todo neurônio da camada seguinte
- ✅ A **função de ativação** entre camadas é OBRIGATÓRIA — sem ela, empilhar camadas lineares produz apenas uma única transformação linear (inútil!)
- ✅ **Teorema da Aproximação Universal:** um MLP com pelo menos 1 camada oculta e neurônios suficientes pode aproximar *qualquer função contínua* — isso explica o poder das redes neurais!

## Funções de ativação

| Função | Fórmula | Intervalo | Quando usar |
|--------|---------|-----------|-------------|
| **Sigmoid** | $1/(1+e^{-z})$ | (0, 1) | Saída de classificação binária |
| **Tanh** | $(e^z - e^{-z})/(e^z + e^{-z})$ | (-1, 1) | Camadas ocultas (era popular) |
| **ReLU** | $\max(0, z)$ | $[0, +\infty)$ | **Default para camadas ocultas** |
| **Leaky ReLU** | $\max(0.01z, z)$ | $(-\infty, +\infty)$ | Quando ReLU morre (dying ReLU) |
| **Softmax** | $e^{z_i}/\sum_j e^{z_j}$ | (0, 1), soma=1 | Saída de classificação multiclasse |

In [ ]:
# ===== Visualizando as funções de ativação =====

z = torch.linspace(-4, 4, 200)  # valores de pré-ativação (entrada da função)

# Calcula cada ativação
sigmoid = torch.sigmoid(z)       # 1 / (1 + e^-z)
tanh    = torch.tanh(z)          # (e^z - e^-z) / (e^z + e^-z)
relu    = F.relu(z)              # max(0, z)
leaky   = F.leaky_relu(z, 0.1)  # max(0.1*z, z)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
z_np = z.numpy()

configs = [
    (axes[0, 0], sigmoid.numpy(), 'Sigmoid  f(z) = 1/(1+e⁻ᶻ)', 'steelblue',
     'Intervalo (0,1). Satura para valores grandes/pequenos → vanishing gradient!'),
    (axes[0, 1], tanh.numpy(),    'Tanh  f(z) = (eᶻ−e⁻ᶻ)/(eᶻ+e⁻ᶻ)', 'darkorange',
     'Intervalo (-1,1). Centralizado em 0 → melhor que sigmoid, mas ainda satura.'),
    (axes[1, 0], relu.numpy(),    'ReLU  f(z) = max(0, z)', 'forestgreen',
     'Sem saturação para z>0 → gradiente não desaparece. Padrão atual.'),
    (axes[1, 1], leaky.numpy(),   'Leaky ReLU  f(z) = max(0.1z, z)', 'crimson',
     'Evita o "neurônio morto": garante gradiente ≠ 0 mesmo para z<0.')
]

for ax, ativ, titulo, cor, desc in configs:
    ax.plot(z_np, ativ, color=cor, linewidth=2.5)
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_title(titulo, fontweight='bold')
    ax.set_xlabel('z (pré-ativação)'); ax.set_ylabel('f(z)')
    ax.grid(True, alpha=0.3)
    # Adiciona nota explicativa abaixo do gráfico
    ax.set_xlabel(f'z\n\n📝 {desc}', fontsize=8)

plt.suptitle('Funções de Ativação', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### ⚠️ O problema do Vanishing Gradient

Funções como **Sigmoid** e **Tanh** *saturam*: para valores muito grandes ou muito pequenos de $z$, a derivada fica **próxima de zero**.

Durante o backpropagation, os gradientes se multiplicam camada a camada. Se cada gradiente é < 1, após muitas camadas o gradiente fica **infinitesimalmente pequeno** — as primeiras camadas quase não aprendem!

$$\frac{\partial L}{\partial W^1} = \frac{\partial L}{\partial W^L} \cdot \frac{\partial W^L}{\partial W^{L-1}} \cdot \ldots \cdot \frac{\partial W^2}{\partial W^1} \to \approx 0$$

**A ReLU resolveu isso**: sua derivada é simplesmente 1 para $z > 0$ — o gradiente passa sem encolher!

In [ ]:
# ===== Demonstração visual do vanishing gradient =====

# Derivadas das funções de ativação
z = torch.linspace(-4, 4, 200, requires_grad=False)

def derivada(func, z):
    """Calcula a derivada de uma função de ativação numericamente."""
    z_ = z.clone().requires_grad_(True)
    saida = func(z_).sum()
    saida.backward()
    return z_.grad.detach()

d_sigmoid = derivada(torch.sigmoid, z)
d_tanh    = derivada(torch.tanh, z)
d_relu    = derivada(F.relu, z)

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(z.numpy(), d_sigmoid.numpy(), label="Derivada da Sigmoid", color='steelblue', linewidth=2)
ax.plot(z.numpy(), d_tanh.numpy(),    label="Derivada da Tanh",    color='darkorange', linewidth=2)
ax.plot(z.numpy(), d_relu.numpy(),    label="Derivada da ReLU",    color='forestgreen', linewidth=2)

ax.axhline(0, color='black', linewidth=0.8)
ax.axhline(1, color='gray',  linewidth=0.8, linestyle='--', label='Gradiente = 1 (ideal)')
ax.fill_between(z.numpy(), d_sigmoid.numpy(), alpha=0.1, color='steelblue')

ax.set_xlabel('z (pré-ativação)')
ax.set_ylabel('Derivada f\'(z)')
ax.set_title('Derivadas das funções de ativação\n'
             '→ Sigmoid/Tanh saturam (derivada → 0) | ReLU mantém gradiente = 1 para z > 0')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.1, 1.1)
plt.tight_layout()
plt.show()

print('💡 Repare: a derivada da Sigmoid nunca passa de 0.25!')
print('   Em 10 camadas: 0.25^10 ≈', 0.25**10, '← gradiente praticamente zero')

---
# 📐 Parte 3 — Backpropagation: como a rede aprende

## A intuição por trás do backprop

A rede aprende ajustando os pesos para **minimizar a função de loss**. O algoritmo para isso é o **Gradient Descent** (descida do gradiente):

$$W \leftarrow W - \alpha \cdot \frac{\partial L}{\partial W}$$

Onde $\alpha$ é a **taxa de aprendizado** (learning rate) e $\frac{\partial L}{\partial W}$ é o gradiente da loss em relação ao peso.

O **Backpropagation** é o algoritmo que calcula esses gradientes de forma eficiente usando a **regra da cadeia** do cálculo:

$$\frac{\partial L}{\partial W^1} = \frac{\partial L}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial a^2} \cdot \frac{\partial a^2}{\partial z^2} \cdot \frac{\partial z^2}{\partial a^1} \cdot \frac{\partial a^1}{\partial W^1}$$

Isso permite calcular os gradientes da **última camada até a primeira** — *propagação para trás*!

### O PyTorch faz isso automaticamente com o **Autograd** 🪄

Você não precisa derivar nada na mão. O PyTorch constrói um **grafo computacional** na memória durante o `forward pass`, e ao chamar `.backward()` ele percorre esse grafo de trás para frente calculando todos os gradientes.

In [ ]:
# ===== Demonstrando o Autograd do PyTorch =====

# Criamos tensores com requires_grad=True para rastrear operações
# O PyTorch vai montar o grafo computacional automaticamente
w = torch.tensor([2.0], requires_grad=True)   # peso
b = torch.tensor([1.0], requires_grad=True)   # bias
x = torch.tensor([3.0])                        # entrada (NÃO precisa de grad)
y_true = torch.tensor([10.0])                  # target

# Forward pass — PyTorch registra cada operação
z = w * x + b                         # pré-ativação: z = 2*3 + 1 = 7
y_hat = torch.relu(z)                  # ativação: relu(7) = 7
loss = (y_hat - y_true) ** 2           # MSE loss: (7-10)^2 = 9

print('=== Forward Pass ===')
print(f'z     = w*x + b = {z.item():.1f}')
print(f'ŷ     = relu(z) = {y_hat.item():.1f}')
print(f'loss  = (ŷ-y)²  = {loss.item():.1f}')

# Backward pass — calcula todos os gradientes automaticamente!
loss.backward()

print('\n=== Backward Pass (gradientes calculados automaticamente) ===')
print(f'∂L/∂w = {w.grad.item():.1f}  (esperado: 2*(ŷ-y)*x = 2*(7-10)*3 = -18)')
print(f'∂L/∂b = {b.grad.item():.1f}  (esperado: 2*(ŷ-y)*1 = 2*(7-10)*1 = -6)')

print('\n=== Atualização dos pesos (SGD com lr=0.01) ===')
lr = 0.01
with torch.no_grad():  # torch.no_grad() para não rastrear essas operações
    w_novo = w - lr * w.grad
    b_novo = b - lr * b.grad
print(f'w antigo: {w.item():.2f} → w novo: {w_novo.item():.2f}')
print(f'b antigo: {b.item():.2f} → b novo: {b_novo.item():.2f}')
print('\n💡 Os pesos foram ajustados na direção que reduz a loss!')

In [ ]:
# ===== Visualizando a descida do gradiente =====
# Problema 1D simples: minimizar f(w) = w² + 2w + 1

def f(w): return w**2 + 2*w + 1  # mínimo em w = -1

w_vals = np.linspace(-4, 2, 200)
f_vals = f(w_vals)

# Simula o gradient descent
w_atual = torch.tensor([3.0], requires_grad=True)  # começa em w=3
lr = 0.3
historico_w = [w_atual.item()]
historico_f = [f(torch.tensor(3.0)).item()]

for _ in range(10):
    if w_atual.grad is not None:
        w_atual.grad.zero_()       # zera o gradiente antes de recalcular
    perda = f(w_atual)
    perda.backward()               # calcula df/dw
    with torch.no_grad():
        w_atual -= lr * w_atual.grad  # w = w - lr * gradiente
    historico_w.append(w_atual.item())
    historico_f.append(f(w_atual).item())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Plot 1: trajetória na curva de loss
axes[0].plot(w_vals, f_vals, 'b-', linewidth=2, label='f(w) = w²+2w+1')
axes[0].scatter(historico_w, historico_f, c=range(len(historico_w)),
                cmap='Reds', zorder=5, s=80)
for i in range(len(historico_w)-1):
    axes[0].annotate('', xy=(historico_w[i+1], historico_f[i+1]),
                     xytext=(historico_w[i], historico_f[i]),
                     arrowprops=dict(arrowstyle='->', color='red'))
axes[0].axvline(-1, color='green', linestyle='--', label='Mínimo (w=-1)')
axes[0].set_xlabel('w'); axes[0].set_ylabel('f(w)')
axes[0].set_title('Descida do Gradiente')
axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[0].annotate('Ponto\ninicial', (historico_w[0], historico_f[0]),
                 xytext=(2, 12), arrowprops=dict(arrowstyle='->'), fontsize=9)

# Plot 2: convergência de w ao longo das iterações
axes[1].plot(historico_w, 'o-', color='red')
axes[1].axhline(-1, color='green', linestyle='--', label='Ótimo (w=-1)')
axes[1].set_xlabel('Iteração'); axes[1].set_ylabel('Valor de w')
axes[1].set_title('Convergência do peso')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'w final após 10 passos: {historico_w[-1]:.4f}  (ótimo = -1.0)')

---
# 🧪 Parte 4 — Laboratório PyTorch: treinando MLP no MNIST

Agora vamos treinar um MLP de verdade! O **MNIST** é o "hello world" do deep learning: 70.000 imagens de dígitos manuscritos (0–9), em escala de cinza, 28×28 pixels.

Para um MLP, vamos **achatar** cada imagem 28×28 em um vetor de 784 valores — o MLP trabalha com vetores, não com a estrutura 2D da imagem (isso é o que as CNNs exploram!).

In [ ]:
# ===== Carregando o MNIST =====

# Transformações:
# 1. ToTensor: converte PIL Image (0–255) para tensor (0.0–1.0)
# 2. Normalize: (valor - média) / desvio_padrão → centraliza os dados
#    Média e desvio do MNIST: 0.1307 e 0.3081
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# Baixa o dataset (primeira vez faz o download, depois usa cache)
mnist_treino = datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
mnist_teste  = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# DataLoaders: dividem o dataset em batches
trainloader = DataLoader(mnist_treino, batch_size=128, shuffle=True,  num_workers=2)
testloader  = DataLoader(mnist_teste,  batch_size=128, shuffle=False, num_workers=2)

print(f'Imagens de treino: {len(mnist_treino)}')
print(f'Imagens de teste:  {len(mnist_teste)}')
print(f'Shape de uma imagem: {mnist_treino[0][0].shape}')  # [1, 28, 28] — 1 canal, 28x28

In [ ]:
# ===== Visualizando algumas imagens do MNIST =====

fig, axes = plt.subplots(2, 10, figsize=(15, 4))

# Linha de cima: imagens
# Linha de baixo: histogramas dos pixels (para ver a distribuição)
for i in range(10):
    img, label = mnist_treino[i]
    img_np = img.squeeze().numpy()  # remove o canal: [1,28,28] → [28,28]

    axes[0, i].imshow(img_np, cmap='gray')
    axes[0, i].set_title(f'Dígito: {label}', fontsize=9)
    axes[0, i].axis('off')

    axes[1, i].hist(img_np.ravel(), bins=20, color='steelblue', edgecolor='none')
    axes[1, i].set_xlabel('Intensidade', fontsize=7)
    axes[1, i].tick_params(labelsize=6)

plt.suptitle('MNIST: imagens e distribuição de pixels por amostra', fontweight='bold')
plt.tight_layout()
plt.show()

print('💡 Cada imagem 28x28 = 784 pixels. Para o MLP, vamos achatar em um vetor de 784 valores.')

## 🔨 Construindo o MLP em PyTorch

Vamos construir três variações do MLP para comparar o efeito da profundidade:

| Modelo | Arquitetura | Parâmetros |
|--------|------------|------------|
| **Raso** | 784 → 128 → 10 | ~101k |
| **Médio** | 784 → 512 → 256 → 10 | ~536k |
| **Profundo** | 784 → 512 → 256 → 128 → 64 → 10 | ~572k |

In [ ]:
class MLP(nn.Module):
    """
    MLP genérico e configurável.
    Recebe uma lista de tamanhos de camadas e monta automaticamente.

    Exemplo:
        MLP([784, 512, 256, 10])  →  784→512→256→10
    """

    def __init__(self, tamanhos, dropout=0.0):
        """
        Args:
            tamanhos: lista de inteiros [entrada, oculta1, oculta2, ..., saída]
            dropout: probabilidade de dropout (0 = sem dropout)
        """
        super().__init__()

        camadas = []  # vai acumular as camadas do modelo

        # Itera pelos pares consecutivos de tamanhos para criar cada camada
        # Ex: [784, 512, 256, 10] → pares: (784,512), (512,256), (256,10)
        for i in range(len(tamanhos) - 1):
            # Camada linear (totalmente conectada)
            camadas.append(nn.Linear(tamanhos[i], tamanhos[i + 1]))

            # Adiciona ativação e regularização em TODAS as camadas, EXCETO a última
            # A última camada é a saída — vai para a CrossEntropyLoss sem ativação
            if i < len(tamanhos) - 2:
                camadas.append(nn.ReLU())             # ativação não-linear
                if dropout > 0:                        # dropout opcional
                    camadas.append(nn.Dropout(dropout))

        # nn.Sequential cria um container que aplica as camadas em sequência
        self.rede = nn.Sequential(*camadas)

    def forward(self, x):
        """
        x: tensor [batch, 1, 28, 28]
        Retorna logits [batch, 10] — um valor por classe
        """
        # ACHATA a imagem 2D em vetor 1D para o MLP processar
        # x.size(0) = tamanho do batch; -1 = calcula automaticamente (= 1*28*28 = 784)
        x = x.view(x.size(0), -1)  # [batch, 1, 28, 28] → [batch, 784]
        return self.rede(x)         # passa pela rede e retorna [batch, 10]


# Cria os três modelos
mlp_raso    = MLP([784, 128, 10])
mlp_medio   = MLP([784, 512, 256, 10])
mlp_profundo = MLP([784, 512, 256, 128, 64, 10])

# Exibe a arquitetura e conta parâmetros
for nome, modelo in [('Raso', mlp_raso), ('Médio', mlp_medio), ('Profundo', mlp_profundo)]:
    total = sum(p.numel() for p in modelo.parameters())
    print(f'MLP {nome:8s} | Parâmetros: {total:>8,} | Arquitetura: {modelo.rede}')
    print()

# Teste rápido: verifica se as dimensões estão corretas
entrada_teste = torch.randn(4, 1, 28, 28)   # batch de 4 imagens
saida_teste   = mlp_medio(entrada_teste)
print(f'Shape de entrada: {entrada_teste.shape}')
print(f'Shape de saída:   {saida_teste.shape}')  # [4, 10] — 4 amostras, 10 classes

In [ ]:
# ===== Função de treinamento (mesma estrutura do notebook de CNNs) =====

def treinar(modelo, trainloader, testloader, epochs=5, lr=0.001, nome='modelo'):
    """
    Treina um modelo e retorna o histórico de métricas por época.

    Args:
        modelo: rede neural (nn.Module)
        trainloader, testloader: DataLoaders
        epochs: número de épocas
        lr: learning rate
        nome: nome para identificar o modelo nos logs

    Returns:
        dict com 'train_loss' e 'test_acc' por época
    """
    modelo = modelo.to(device)

    # CrossEntropyLoss: combina LogSoftmax + NLLLoss
    # Recebe logits (saída bruta sem ativação) e inteiros (classes)
    criterio = nn.CrossEntropyLoss()

    # Adam: adapta a lr para cada parâmetro individualmente → converge rápido
    otimizador = optim.Adam(modelo.parameters(), lr=lr)

    historico = {'train_loss': [], 'test_acc': []}

    for epoch in range(epochs):

        # ---- Modo treino ----
        # modelo.train() ativa Dropout (probabilidades reais) e
        # BatchNorm usa estatísticas do batch atual
        modelo.train()
        loss_acum = 0.0
        n = 0

        for imgs, labels in trainloader:
            imgs, labels = imgs.to(device), labels.to(device)

            otimizador.zero_grad()          # 1. Zera gradientes acumulados
            saidas = modelo(imgs)           # 2. Forward: calcula logits
            loss = criterio(saidas, labels) # 3. Calcula a loss
            loss.backward()                 # 4. Backward: calcula gradientes
            otimizador.step()               # 5. Atualiza os pesos

            loss_acum += loss.item() * imgs.size(0)  # acumula loss ponderada
            n += imgs.size(0)

        loss_media = loss_acum / n

        # ---- Modo avaliação ----
        # modelo.eval() desativa Dropout e usa estatísticas salvas do BatchNorm
        modelo.eval()
        acertos = total = 0

        with torch.no_grad():  # desativa o autograd → mais rápido, menos memória
            for imgs, labels in testloader:
                imgs, labels = imgs.to(device), labels.to(device)
                saidas = modelo(imgs)
                _, preditos = torch.max(saidas, dim=1)  # classe com maior logit
                acertos += (preditos == labels).sum().item()
                total += labels.size(0)

        acuracia = 100.0 * acertos / total
        historico['train_loss'].append(loss_media)
        historico['test_acc'].append(acuracia)

        print(f'[{nome}] Época {epoch+1:2d}/{epochs} | '
              f'Loss: {loss_media:.4f} | Acurácia: {acuracia:.2f}%')

    return historico

In [ ]:
# ===== Treina os 3 modelos e compara =====

print('🚀 Treinando MLP Raso...')
hist_raso = treinar(MLP([784, 128, 10]).to(device),
                    trainloader, testloader, epochs=5, nome='Raso')

print('\n🚀 Treinando MLP Médio...')
hist_medio = treinar(MLP([784, 512, 256, 10]).to(device),
                     trainloader, testloader, epochs=5, nome='Médio')

print('\n🚀 Treinando MLP Profundo...')
hist_profundo = treinar(MLP([784, 512, 256, 128, 64, 10]).to(device),
                        trainloader, testloader, epochs=5, nome='Profundo')

In [ ]:
# ===== Comparando os 3 modelos visualmente =====

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epocas = range(1, 6)

# Plot de loss
for hist, nome, cor in [
    (hist_raso,    'Raso (784→128→10)',           'steelblue'),
    (hist_medio,   'Médio (784→512→256→10)',       'darkorange'),
    (hist_profundo,'Profundo (784→512→256→128→64→10)', 'forestgreen')
]:
    axes[0].plot(epocas, hist['train_loss'], 'o-', color=cor, label=nome)
    axes[1].plot(epocas, hist['test_acc'],   'o-', color=cor, label=nome)

axes[0].set_xlabel('Época'); axes[0].set_ylabel('Loss de treino')
axes[0].set_title('Comparação de Loss'); axes[0].legend(); axes[0].grid(True)

axes[1].set_xlabel('Época'); axes[1].set_ylabel('Acurácia no teste (%)')
axes[1].set_title('Comparação de Acurácia no Teste'); axes[1].legend(); axes[1].grid(True)

plt.tight_layout()
plt.show()

print(f'\nAcurácias finais:')
print(f'  Raso:     {hist_raso["test_acc"][-1]:.2f}%')
print(f'  Médio:    {hist_medio["test_acc"][-1]:.2f}%')
print(f'  Profundo: {hist_profundo["test_acc"][-1]:.2f}%')

In [ ]:
# ===== Visualizando os pesos aprendidos pela primeira camada =====
#
# Os pesos da 1ª camada (784 → 128) têm shape [128, 784].
# Cada linha é o "filtro" de um neurônio — 784 pesos, um por pixel.
# Se redimensionarmos para 28x28, vemos o PADRÃO que aquele neurônio detecta!

# Recria e treina o MLP médio para ter o modelo salvo
modelo_final = MLP([784, 256, 128, 10])
treinar(modelo_final, trainloader, testloader, epochs=5, nome='Final', lr=0.001)

# Pega os pesos da primeira camada linear
# modelo_final.rede[0] é a primeira nn.Linear
pesos_primeira_camada = modelo_final.rede[0].weight.data  # shape: [256, 784]

# Visualiza os 20 primeiros neurônios
fig, axes = plt.subplots(4, 5, figsize=(12, 10))
for i, ax in enumerate(axes.ravel()):
    w = pesos_primeira_camada[i].reshape(28, 28).cpu().numpy()  # reformata para imagem
    im = ax.imshow(w, cmap='RdBu', vmin=-w.max(), vmax=w.max())
    ax.set_title(f'Neurônio {i+1}', fontsize=8)
    ax.axis('off')

plt.suptitle('Pesos da 1ª camada — o que cada neurônio "procura" na imagem\n'
             '(Vermelho = ativa com pixel claro, Azul = ativa com pixel escuro)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ===== Matriz de Confusão: quais dígitos o modelo confunde? =====

# Coleta todas as predições no conjunto de teste
modelo_final.eval()
modelo_final.to(device)

todos_pred, todos_real = [], []

with torch.no_grad():
    for imgs, labels in testloader:
        imgs = imgs.to(device)
        saidas = modelo_final(imgs)
        _, preditos = torch.max(saidas, dim=1)
        todos_pred.extend(preditos.cpu().numpy())
        todos_real.extend(labels.numpy())

# Calcula a matriz de confusão manualmente
todos_pred = np.array(todos_pred)
todos_real = np.array(todos_real)

conf_matrix = np.zeros((10, 10), dtype=int)
for real, pred in zip(todos_real, todos_pred):
    conf_matrix[real][pred] += 1  # linha = classe real, coluna = classe prevista

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(conf_matrix, cmap='Blues')

# Adiciona os números dentro de cada célula
for i in range(10):
    for j in range(10):
        cor = 'white' if conf_matrix[i, j] > conf_matrix.max() / 2 else 'black'
        ax.text(j, i, str(conf_matrix[i, j]), ha='center', va='center',
                color=cor, fontsize=9)

ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xticklabels(range(10)); ax.set_yticklabels(range(10))
ax.set_xlabel('Classe Prevista', fontsize=12)
ax.set_ylabel('Classe Real', fontsize=12)
ax.set_title('Matriz de Confusão — MNIST\n'
             'Diagonal principal = acertos | Fora = erros', fontsize=12)
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

# Identifica os pares mais confundidos
matriz_erros = conf_matrix.copy()
np.fill_diagonal(matriz_erros, 0)  # zera a diagonal (acertos)
idx = np.unravel_index(np.argsort(matriz_erros.ravel())[-3:], (10, 10))
print('Pares de dígitos mais confundidos:')
for real, pred in zip(idx[0][::-1], idx[1][::-1]):
    print(f'  Dígito {real} confundido com {pred}: {matriz_erros[real, pred]} vezes')

---
# 🛡️ Parte 5 — Regularização e Técnicas de Otimização

## Overfitting vs. Underfitting

```
Underfitting            Bom ajuste            Overfitting
─────────────        ─────────────          ─────────────
Loss treino alta     Loss baixa             Loss treino bem baixa
Loss teste alta      Loss teste próxima     Loss teste MUITO maior
                     da loss de treino

→ Modelo muito        → Generaliza bem       → Decorou o treino,
  simples ou                                    não generaliza
  pouco treinado
```

## Técnicas de regularização

| Técnica | O que faz | Quando usar |
|---------|-----------|-------------|
| **Dropout** | Zera aleatoriamente p% dos neurônios no treino | Overfitting em camadas densas |
| **Batch Normalization** | Normaliza as ativações dentro do batch | Quase sempre! Estabiliza treino |
| **Weight Decay (L2)** | Penaliza pesos grandes na loss | Overfitting moderado |
| **Early Stopping** | Para o treino quando val loss para de cair | Sempre que possível |

In [ ]:
# ===== Comparando: sem regularização vs. com Dropout vs. com BatchNorm =====

class MLPComRegularizacao(nn.Module):
    """MLP com suporte a BatchNorm e Dropout para experimentos."""

    def __init__(self, usar_batchnorm=False, usar_dropout=False, p_dropout=0.5):
        super().__init__()

        camadas = []

        # Bloco 1: 784 → 512
        camadas.append(nn.Linear(784, 512))
        if usar_batchnorm:
            # BatchNorm1d: normaliza cada feature do batch
            # Aprende escala (gamma) e deslocamento (beta) — 2*512 parâmetros
            camadas.append(nn.BatchNorm1d(512))
        camadas.append(nn.ReLU())
        if usar_dropout:
            # Dropout: durante o treino, zera p_dropout% dos neurônios aleatoriamente
            # No teste, usa TODOS os neurônios (com pesos escalonados)
            camadas.append(nn.Dropout(p_dropout))

        # Bloco 2: 512 → 256
        camadas.append(nn.Linear(512, 256))
        if usar_batchnorm:
            camadas.append(nn.BatchNorm1d(256))
        camadas.append(nn.ReLU())
        if usar_dropout:
            camadas.append(nn.Dropout(p_dropout))

        # Saída
        camadas.append(nn.Linear(256, 10))

        self.rede = nn.Sequential(*camadas)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.rede(x)


print('🔬 Experimento: efeito de Dropout e BatchNorm')
print('='*60)

resultados = {}
configs = [
    ('Sem regularização', MLPComRegularizacao()),
    ('Com Dropout(0.5)',  MLPComRegularizacao(usar_dropout=True)),
    ('Com BatchNorm',     MLPComRegularizacao(usar_batchnorm=True)),
    ('Dropout + BN',      MLPComRegularizacao(usar_batchnorm=True, usar_dropout=True)),
]

for nome, modelo in configs:
    print(f'\n▶ {nome}')
    hist = treinar(modelo, trainloader, testloader, epochs=5, lr=0.001, nome=nome)
    resultados[nome] = hist

# Visualiza
fig, ax = plt.subplots(figsize=(10, 5))
cores = ['steelblue', 'darkorange', 'forestgreen', 'crimson']
for (nome, _), cor in zip(configs, cores):
    ax.plot(range(1, 6), resultados[nome]['test_acc'], 'o-', color=cor, label=nome)

ax.set_xlabel('Época'); ax.set_ylabel('Acurácia no teste (%)')
ax.set_title('Efeito de Dropout e BatchNorm na Acurácia')
ax.legend(); ax.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# ===== Comparando otimizadores: SGD vs. Adam vs. RMSprop =====

# Todos os modelos têm a mesma arquitetura — só o otimizador muda
def treinar_com_otimizador(nome_opt, opt_class, opt_kwargs, epochs=5):
    modelo = MLP([784, 256, 128, 10]).to(device)
    criterio = nn.CrossEntropyLoss()
    otimizador = opt_class(modelo.parameters(), **opt_kwargs)

    historico = {'train_loss': [], 'test_acc': []}

    for epoch in range(epochs):
        modelo.train()
        loss_acum, n = 0.0, 0
        for imgs, labels in trainloader:
            imgs, labels = imgs.to(device), labels.to(device)
            otimizador.zero_grad()
            loss = criterio(modelo(imgs), labels)
            loss.backward()
            otimizador.step()
            loss_acum += loss.item() * imgs.size(0)
            n += imgs.size(0)

        modelo.eval()
        acertos = total = 0
        with torch.no_grad():
            for imgs, labels in testloader:
                imgs, labels = imgs.to(device), labels.to(device)
                _, pred = torch.max(modelo(imgs), 1)
                acertos += (pred == labels).sum().item()
                total += labels.size(0)

        historico['train_loss'].append(loss_acum / n)
        historico['test_acc'].append(100.0 * acertos / total)
        print(f'[{nome_opt}] Época {epoch+1}/{epochs} | '
              f'Loss: {historico["train_loss"][-1]:.4f} | '
              f'Acc: {historico["test_acc"][-1]:.2f}%')
    return historico


# Três otimizadores clássicos
print('\n--- SGD (Stochastic Gradient Descent) ---')
# SGD puro: w = w - lr * grad. Simples mas pode oscilar.
# momentum=0.9 acumula velocidade na direção do gradiente → converge mais rápido
hist_sgd = treinar_com_otimizador('SGD+momentum', optim.SGD,
                                   {'lr': 0.01, 'momentum': 0.9})

print('\n--- Adam (Adaptive Moment Estimation) ---')
# Adam: adapta a lr por parâmetro usando média e variância dos gradientes.
# lr=0.001 é um ótimo default — geralmente não precisa ajustar.
hist_adam = treinar_com_otimizador('Adam', optim.Adam, {'lr': 0.001})

print('\n--- RMSprop ---')
# RMSprop: divide a lr pela raiz da média quadrática dos gradientes recentes.
# Bom para redes recorrentes (RNNs).
hist_rms = treinar_com_otimizador('RMSprop', optim.RMSprop, {'lr': 0.001})

# Comparação visual
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epocas = range(1, 6)
for hist, nome, cor in [
    (hist_sgd,  'SGD + momentum', 'steelblue'),
    (hist_adam, 'Adam',           'darkorange'),
    (hist_rms,  'RMSprop',        'forestgreen')
]:
    axes[0].plot(epocas, hist['train_loss'], 'o-', color=cor, label=nome)
    axes[1].plot(epocas, hist['test_acc'],   'o-', color=cor, label=nome)

axes[0].set_title('Loss de Treino por Otimizador'); axes[0].legend(); axes[0].grid(True)
axes[1].set_title('Acurácia no Teste por Otimizador'); axes[1].legend(); axes[1].grid(True)
for ax in axes:
    ax.set_xlabel('Época')
plt.tight_layout()
plt.show()

---
# 📝 Resumo do que aprendemos

✅ **Neurônio artificial**: combinação linear ($z = Wx + b$) + função de ativação ($f(z)$)

✅ **Perceptron**: 1 neurônio → fronteira linear. Não resolve XOR → precisamos de camadas!

✅ **Funções de ativação**: ReLU é o default atual — sem saturação, gradiente saudável

✅ **Backpropagation**: regra da cadeia aplicada de trás pra frente — o PyTorch faz isso automaticamente

✅ **Gradient Descent**: $W \leftarrow W - \alpha \nabla_W L$ — iterativamente reduz a loss

✅ **Loop de treino**: `zero_grad → forward → loss → backward → step`

✅ **Regularização**: Dropout e BatchNorm reduzem overfitting e estabilizam o treino

✅ **Otimizadores**: Adam é o default confiável; SGD com momentum pode superar com tuning certo

### MLP vs. CNN: qual a diferença?

| | **MLP** | **CNN** |
|---|---------|-------|
| Entrada | Vetor achatado | Imagem 2D (preserva estrutura espacial) |
| Parâmetros | Muitos (cada pixel conecta a cada neurônio) | Poucos (pesos compartilhados no filtro) |
| Invariância | Nenhuma | Translação parcial (via pooling) |
| Uso | Dados tabulares, embeddings | Imagens, sinais 1D/2D |

---
# 🎯 Exercício Final

Agora é a sua vez! Você vai aplicar tudo que aprendeu no dataset **FashionMNIST** — igual ao MNIST em formato, mas com imagens de roupas e calçados (muito mais desafiador!).

**As 10 classes:** T-shirt/top, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, Ankle boot

---

### Parte A — MLP Básico (30 pontos)

1. Carregue o `datasets.FashionMNIST` com as transformações adequadas (28×28, 1 canal, normalização `mean=0.2860, std=0.3530`).
2. Construa um MLP com **pelo menos 3 camadas ocultas** usando a classe `MLP` definida na aula.
3. Treine por **5 épocas** e reporte a acurácia final.
4. Plote as curvas de **loss** e **acurácia**.

> 📌 Referência: um bom MLP deve passar de **88%** no FashionMNIST.

---

### Parte B — Efeito da Regularização (35 pontos)

Treine **3 versões** do mesmo modelo, variando apenas a regularização:

1. **Sem regularização** — arquitetura base
2. **Com Dropout(0.3)**
3. **Com BatchNorm + Dropout(0.3)**

Plote as 3 curvas de acurácia no mesmo gráfico e responda:
- Qual versão teve melhor acurácia?
- Alguma mostrou sinais de overfitting?

---

### Parte C — Análise dos Erros (35 pontos)

Com o melhor modelo da Parte B:

1. Gere a **matriz de confusão** (10×10) para o conjunto de teste.
2. Identifique os **3 pares de classes mais confundidos**.
3. Visualize **5 imagens** onde o modelo errou, mostrando classe real vs. predita.
4. Responda em markdown: por que você acha que essas classes são confundidas?

---

### 💡 Dicas

- Reutilize a função `treinar()` e a classe `MLP` definidas na Parte 4 — não reinvente!
- Use `MLPComRegularizacao` da Parte 5 para facilitar a Parte B.
- Para a matriz de confusão, copie o código da célula acima e ajuste.
- A aula de CNNs tem o código de visualização de erros na Parte C do exercício final daquele notebook — pode reutilizar adaptando!

Boa sorte! 🚀

In [ ]:
# ============================================================
# PARTE A — MLP Básico no FashionMNIST
# ============================================================

# 1) Transformações para FashionMNIST
transform_fashion_treino = transforms.Compose([
    transforms.RandomHorizontalFlip(),            # augmentation: flip horizontal
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,))    # média e std do FashionMNIST
])

transform_fashion_teste = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,))
])

fashion_trainset = datasets.FashionMNIST(root='./data', train=True,
                                          download=True, transform=transform_fashion_treino)
fashion_testset  = datasets.FashionMNIST(root='./data', train=False,
                                          download=True, transform=transform_fashion_teste)

fashion_trainloader = DataLoader(fashion_trainset, batch_size=128, shuffle=True,  num_workers=2)
fashion_testloader  = DataLoader(fashion_testset,  batch_size=128, shuffle=False, num_workers=2)

fashion_classes = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
                   'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print(f'FashionMNIST | Treino: {len(fashion_trainset)} | Teste: {len(fashion_testset)}')


# 2) MLP com 3 camadas ocultas
# TODO: defina a arquitetura — experimente tamanhos!
# Dica: 784 → ??? → ??? → ??? → 10
meu_mlp = MLP([784, 512, 256, 128, 10])

total_params = sum(p.numel() for p in meu_mlp.parameters())
print(f'Parâmetros: {total_params:,}')


# 3) Treina por 5 épocas
print('\n🚀 Treinando MLP no FashionMNIST...')
hist_fashion = treinar(meu_mlp, fashion_trainloader, fashion_testloader,
                       epochs=5, lr=0.001, nome='FashionMLP')


# 4) Curvas de loss e acurácia
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(range(1, 6), hist_fashion['train_loss'], 'o-', color='royalblue')
axes[0].set_xlabel('Época'); axes[0].set_ylabel('Loss')
axes[0].set_title('Parte A — Loss de Treino'); axes[0].grid(True)

axes[1].plot(range(1, 6), hist_fashion['test_acc'], 'o-', color='green')
axes[1].set_xlabel('Época'); axes[1].set_ylabel('Acurácia (%)')
axes[1].set_title('Parte A — Acurácia no Teste'); axes[1].grid(True)

plt.tight_layout(); plt.show()

print(f'\n✅ Acurácia final (Parte A): {hist_fashion["test_acc"][-1]:.2f}%')

In [ ]:
# ============================================================
# PARTE B — Efeito da Regularização
# ============================================================

# Reutiliza MLPComRegularizacao definido na Parte 5, agora no FashionMNIST

configs_b = [
    ('Sem regularização',         MLPComRegularizacao()),
    ('Com Dropout(0.3)',          MLPComRegularizacao(usar_dropout=True, p_dropout=0.3)),
    ('BatchNorm + Dropout(0.3)',  MLPComRegularizacao(usar_batchnorm=True, usar_dropout=True, p_dropout=0.3)),
]

resultados_b = {}
for nome, modelo in configs_b:
    print(f'\n▶ {nome}')
    hist = treinar(modelo, fashion_trainloader, fashion_testloader,
                   epochs=5, lr=0.001, nome=nome)
    resultados_b[nome] = hist

# Gráfico comparativo
fig, ax = plt.subplots(figsize=(10, 5))
cores_b = ['steelblue', 'darkorange', 'forestgreen']
for (nome, _), cor in zip(configs_b, cores_b):
    ax.plot(range(1, 6), resultados_b[nome]['test_acc'], 'o-', color=cor, label=nome)

ax.set_xlabel('Época'); ax.set_ylabel('Acurácia no Teste (%)')
ax.set_title('Parte B — FashionMNIST: Efeito da Regularização')
ax.legend(); ax.grid(True)
plt.tight_layout(); plt.show()

print('\nAcurácias finais:')
for nome, _ in configs_b:
    print(f'  {nome}: {resultados_b[nome]["test_acc"][-1]:.2f}%')

In [ ]:
# ============================================================
# PARTE C — Análise dos Erros
# ============================================================

# Usa o melhor modelo da Parte B
melhor_nome = max(resultados_b, key=lambda k: max(resultados_b[k]['test_acc']))
print(f'Melhor modelo: {melhor_nome}')

# Recupera o modelo treinado correspondente
idx_melhor = [n for n, _ in configs_b].index(melhor_nome)
modelo_melhor = configs_b[idx_melhor][1]
modelo_melhor.eval()
modelo_melhor.to(device)

# 1) Coleta predições
todos_pred, todos_real, todas_imgs = [], [], []

with torch.no_grad():
    for imgs, labels in fashion_testloader:
        imgs_dev = imgs.to(device)
        saidas = modelo_melhor(imgs_dev)
        _, preditos = torch.max(saidas, dim=1)
        todos_pred.extend(preditos.cpu().numpy())
        todos_real.extend(labels.numpy())
        todas_imgs.extend(imgs)  # guarda para visualização de erros

todos_pred = np.array(todos_pred)
todos_real = np.array(todos_real)

# 1) Matriz de confusão
conf = np.zeros((10, 10), dtype=int)
for r, p in zip(todos_real, todos_pred):
    conf[r][p] += 1

fig, ax = plt.subplots(figsize=(12, 9))
im = ax.imshow(conf, cmap='Blues')
for i in range(10):
    for j in range(10):
        cor = 'white' if conf[i, j] > conf.max() / 2 else 'black'
        ax.text(j, i, str(conf[i, j]), ha='center', va='center', color=cor, fontsize=8)

ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xticklabels(fashion_classes, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(fashion_classes, fontsize=9)
ax.set_xlabel('Classe Prevista'); ax.set_ylabel('Classe Real')
ax.set_title(f'Matriz de Confusão — FashionMNIST ({melhor_nome})', fontweight='bold')
plt.colorbar(im); plt.tight_layout(); plt.show()

# 2) Pares mais confundidos
conf_erros = conf.copy()
np.fill_diagonal(conf_erros, 0)
idx = np.unravel_index(np.argsort(conf_erros.ravel())[-3:], (10, 10))
print('\n3 pares mais confundidos:')
for r, p in zip(idx[0][::-1], idx[1][::-1]):
    print(f'  {fashion_classes[r]} → predito como {fashion_classes[p]}: {conf_erros[r, p]} vezes')

# 3) Visualiza 5 erros
mask_erros = todos_pred != todos_real
idxs_erros = np.where(mask_erros)[0][:5]

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for i, idx_err in enumerate(idxs_erros):
    img = todas_imgs[idx_err].squeeze().numpy()
    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(
        f'Real: {fashion_classes[todos_real[idx_err]]}\n'
        f'Pred: {fashion_classes[todos_pred[idx_err]]}',
        fontsize=8, color='red'
    )
    axes[i].axis('off')

plt.suptitle('Parte C — 5 erros do modelo', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## Parte C — Análise (preencha aqui)

**1. Qual versão teve melhor acurácia? Por quê?**

> *Responda aqui...*

**2. Algum modelo mostrou sinais de overfitting?**

> *Responda aqui...*

**3. Por que as classes mais confundidas são confundidas?**

> *Responda aqui...*

---

**Lembre-se:** No MLP, cada pixel é tratado independentemente — a rede não sabe que pixels vizinhos formam bordas ou texturas. Por isso o MLP sempre perde para uma CNN em dados de imagem. Na próxima aula, você vai ver como as CNNs exploram a estrutura espacial para superar o MLP em imagens! 🚀